# Workday Jobs Scraper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/workday-jobs-scraper.ipynb)

**What:** Workday jobs from 165+ verified employers or any myworkdayjobs.com site, verified by 64 passing tests. Keyword, location and posted-date filters, full descriptions, salary ranges, remote/hybrid, and only-new-jobs delta mode. $1 per 1,000 jobs.

**Why:** run the [`workday-jobs-scraper`](https://apify.com/transparent_meteorite/workday-jobs-scraper) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/workday-jobs-scraper) for every option.

In [2]:
run_input = {'companies': ['NVIDIA', 'Salesforce'], 'searchText': 'engineer', 'maxItemsPerCompany': 5}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/workday-jobs-scraper').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.workday-jobs-scraper runId:gFrzZiRBOp26ptPTF] -> Status: RUNNING, Message: 


[apify.workday-jobs-scraper runId:gFrzZiRBOp26ptPTF] -> 2026-10-13T22:20:53.752Z ACTOR: Pulling container image of build lstuBqce824JlQyza from registry.


[apify.workday-jobs-scraper runId:gFrzZiRBOp26ptPTF] -> 2026-10-13T22:20:53.754Z ACTOR: Creating container.


[apify.workday-jobs-scraper runId:gFrzZiRBOp26ptPTF] -> 2026-10-13T22:20:53.815Z ACTOR: Starting container.


[apify.workday-jobs-scraper runId:gFrzZiRBOp26ptPTF] -> 2026-10-13T22:20:53.816Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.workday-jobs-scraper runId:gFrzZiRBOp26ptPTF] -> 2026-10-13T22:20:58.984Z Done: {"sites":2,"scanned":10,"emitted":10,"charged":10,"skippedSeen":0,"unknownCompanies":[],"invalidUrls":[],"siteErrors":0,"chargeLimitReached":false,"warnings":0}


[apify.workday-jobs-scraper runId:gFrzZiRBOp26ptPTF] -> 2026-10-13T22:20:58.986Z INFO  [Status message]: Done: 10 items


[apify.workday-jobs-scraper runId:gFrzZiRBOp26ptPTF] -> 2026-10-13T22:20:59.044Z Thanks for using Workday Jobs Scraper. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/Cu3845eApZunSm9ge/issues) with the run URL and we will reply there.


[apify.workday-jobs-scraper runId:gFrzZiRBOp26ptPTF] -> Status: SUCCEEDED, Message: Done: 10 items


Run status: SUCCEEDED | items: 10


## 4. Tabulate

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["company", "title", "location", "country", "remoteType", "timeType", "postedOn", "salaryMin", "salaryMax", "salaryCurrency"] if c in df.columns]
df[SHOW]

,company,title,location,country,remoteType,timeType,postedOn,salaryMin,salaryMax,salaryCurrency
0,Salesforce,"Director, Product Management - B2B Revenue Automation",California - San Francisco,NaN,NaN,NaN,2026-10-13,NaN,NaN,NaN
1,Salesforce,Software Engineering Architect - DevXP AI team,California - San Francisco,NaN,NaN,NaN,2026-10-13,NaN,NaN,NaN
2,Salesforce,"Sr. GRC Analyst, Common Control Framework",Washington - Bellevue,United States of America,Office Tech-Flexible,Full time,2026-10-13,117200.0,176700.0,USD
3,Salesforce,"Product Management Director, Platform Services",California - San Francisco,United States of America,Office Tech-Flexible,Full time,2026-10-13,197300.0,313700.0,USD
4,Salesforce,Manager/Sr. Manager - Product Release Marketing (Public Sector),Virginia - Washington DC Metro - Remote,United States of America,Office - Flexible,Full time,2026-10-13,117400.0,216900.0,USD
5,NVIDIA,"Director, Engineering – Software Engineering","Vietnam, Hanoi",Vietnam,NaN,Full time,2026-07-10,NaN,NaN,NaN
6,NVIDIA,"Senior System Engineer, Solution Engineering","Poland, Remote",Poland,Remote,Full time,2026-09-22,NaN,NaN,NaN
7,NVIDIA,"Senior Software Engineer, Test Engineering","China, Beijing",China,NaN,Full time,2026-10-12,NaN,NaN,NaN
8,NVIDIA,"Senior Software Engineer, Agentic Engineering","US, CA, Santa Clara",NaN,NaN,NaN,2026-10-12,NaN,NaN,NaN
9,NVIDIA,"Senior Software Engineer, Agentic Engineering","US, CA, Santa Clara",NaN,NaN,NaN,2026-10-12,NaN,NaN,NaN


## 5. A quick analysis

In [5]:
for c in ['salaryMin','salaryMax']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
print(df.groupby('company').size().rename('roles in sample'))
print(df['remoteType'].value_counts(dropna=False))
df.groupby('company')[['salaryMin','salaryMax']].median()

company
NVIDIA        5
Salesforce    5
Name: roles in sample, dtype: int64
remoteType
NaN                     6
Office Tech-Flexible    2
Office - Flexible       1
Remote                  1
Name: count, dtype: int64


,salaryMin,salaryMax
company,,
NVIDIA,NaN,NaN
Salesforce,117400.0,216900.0


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [workday-jobs-scraper on the Apify Store](https://apify.com/transparent_meteorite/workday-jobs-scraper).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.